# Limiting reagent and percent yield

*chemistry · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/chemistry/limiting_reagent_yield)

Mix aluminium with iodine (or nitrogen with hydrogen) in any proportion: which reactant runs out first, how much product can form, and how much of the other is left over? Every amount moves with one reaction extent $\xi$, the limiting reagent is a minimum over $n_s/|\nu_s|$, and a whole batch of experiments, and a 2D map of starting masses, is handled by the same tensor operation.

**Vector form**

$$\mathbf n(\xi) = \mathbf n^0 + \boldsymbol\nu\,\xi,\qquad \xi^{\max} = \min_{s:\,\nu_s<0}\frac{n^0_s}{|\nu_s|},\qquad m_p^{\mathrm{theo}} = M_p\,\nu_p\,\xi^{\max},\qquad Y = \frac{m_p^{\mathrm{weighed}}}{m_p^{\mathrm{theo}}}$$

**Index (tensor) form**

$$n^0_{bs} = \frac{m^0_{bs}}{M_s},\quad M_s = A_{es}\,m_e \qquad n_{bks} = n^0_{bs} + \nu_s\,\xi_{bk} \qquad \xi^{\max}_b = \min_{s\in R}\frac{n^0_{bs}}{|\nu_s|} \qquad Y_b = \frac{m^{\mathrm w}_b}{M_p\,\nu_p\,\xi^{\max}_b}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\nu_s$ | `nu` | (S,) | signed stoichiometric numbers: $-2, -3, +2$ for 2 Al + 3 I₂ → 2 AlI₃ |
| $M_s = A_{es}m_e$ | `einsum('es,e->s', A, m)` | (S,) | repeated $e$ ⇒ summed: molar masses from the atoms in each species |
| $n^0_{bs}$ | `m0 / molar` | (B, S) | moles at the start for each experiment $b$ (the last one is your run) |
| $\xi^{\max}_b$ | `where(nu < 0, n0 / nu.abs(), inf).min(-1)` | (B,) | largest extent before a reactant hits zero; argmin is the limiting reagent |
| $n_{bks}$ | `n0[:, None] + nu * xi[..., None]` | (B, K, S) | amounts at $K$ extents from 0 to $\xi^{\max}$ (the bars) |
| $M_p\nu_p\xi^{\max}_b$ | `theoretical_mass(xi_max, nu, molar)` | (B,) | theoretical yield of the product in g |
| $Y_b$ | `100 * weighed / theory` | (B,) | percent yield |
| $\xi^{\max}(m^0_1, m^0_2)$ | `yield_map(grid_1, grid_2, nu, molar)` | (G, G) | the grid of starting masses is just a bigger batch |

Moles, not grams, react in fixed ratios: first divide each mass by its molar mass. Then one number, the extent $\xi$, describes how far the reaction has gone: $2\,\mathrm{Al} + 3\,\mathrm{I_2} \rightarrow 2\,\mathrm{AlI_3}$ uses $2\xi$ mol of aluminium and $3\xi$ mol of iodine and makes $2\xi$ mol of aluminium iodide. Each reactant allows at most $n^0_s/|\nu_s|$ units of reaction, and the smallest of these numbers wins: that reactant is the limiting reagent, and its bar is the one that reaches zero. With 1.20 g Al (44.5 mmol) and 2.40 g I₂ (9.46 mmol) iodine is limiting: $\xi^{\max} = 3.15$ mmol, at most 2.57 g of AlI₃, and 1.03 g of aluminium stays behind. On the map the product depends on only one reactant on each side of the dashed stoichiometric line; along the line both run out together and nothing is wasted. Real yields are lower than the theoretical ones (losses, side reactions, an equilibrium that does not go to completion: a single pass through a Haber reactor converts only about 15 %), and a yield above 100 % means the product is wet or impure. Change the masses, the weighed product or the reaction.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/chemistry/limiting_reagent_yield/main.py
"""Main loop: masses → moles → limiting reagent and yields for the batch → extent bars and yield map."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    reaction = params["reaction"]
    nu, molar, species = tensors.stoichiometry(reaction, dev)  # (S,), (S,)
    m0, weighed = tensors.experiments(reaction, params["mass_1"], params["mass_2"], params["measured"], dev)
    res = solve.analyse(m0, weighed, nu, molar)
    print("molar masses: " + ", ".join(f"{s} {M:.3f} g/mol" for s, M in zip(species, molar.tolist())))

    for b in range(len(m0)):
        label = "your run" if b == len(m0) - 1 else f"run {b + 1}"
        lim = res["limiting"][b].item()
        excess = 1 - lim
        print(
            f"{label:<8} {m0[b, 0].item():7.2f} g {species[0]} + {m0[b, 1].item():7.2f} g {species[1]}: "
            f"limiting {species[lim]:<3} ξ_max = {res['xi_max'][b].item():.4g} mol   "
            f"theoretical {res['theory'][b].item():7.3f} g {species[2]}   "
            f"left over {res['leftover_mass'][b, excess].item():6.3f} g {species[excess]}   "
            f"yield {weighed[b].item():.2f} g = {res['percent'][b].item():5.1f} %"
        )
    if res["percent"][-1] > 100:
        print("your run is above 100 %: the product must be wet or impure (or a mass was misread)")

    xi, n_sweep = solve.extent_sweep(res["n0"], nu, res["xi_max"])  # (B, K), (B, K, S)
    ratio = (nu[1].abs() * molar[1]) / (nu[0].abs() * molar[0])  # g of reactant 2 per g of reactant 1
    print(f"stoichiometric mass ratio: {ratio.item():.4f} g {species[1]} per g {species[0]}")
    G = 41 if fast_mode() else 201
    x_max = max(2.0 * params["mass_1"], 2.0 * params["mass_2"] / ratio.item())  # map axes scaled so that the
    grid_1 = torch.linspace(0.0, x_max, G, dtype=torch.float64, device=dev)  # stoichiometric line is the diagonal
    grid_2 = torch.linspace(0.0, x_max * ratio.item(), G, dtype=torch.float64, device=dev)
    product_map = solve.yield_map(grid_1, grid_2, nu, molar)  # (G, G)

    return plot.figure(xi[-1], n_sweep[-1], species, res, weighed, grid_1, grid_2, product_map, ratio, m0, out_dir)


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/chemistry/limiting_reagent_yield/equation.py
r"""Chemistry → tensor translation.

A reaction with signed stoichiometric numbers ν_s (negative for reactants) changes every amount by the same
reaction extent ξ (in mol):

    n_{bs}(ξ) = n⁰_{bs} + ν_s ξ_b                       (experiment b, species s; no index summed)

The reaction stops when the first reactant runs out — the limiting reagent:

    ξ^max_b = min_{s: ν_s < 0}  n⁰_{bs} / |ν_s|

Theoretical yield of the product p: n_{bp} = ν_p ξ^max_b, i.e. a mass M_p ν_p ξ^max_b;
percent yield = weighed / theoretical.
"""

import torch


def moles(mass: torch.Tensor, molar: torch.Tensor) -> torch.Tensor:
    """n_{bs} = m_{bs} / M_s.   (B, S), (S,) → (B, S)."""
    return mass / molar


def amounts(n0: torch.Tensor, nu: torch.Tensor, xi: torch.Tensor) -> torch.Tensor:
    """n_{b…s}(ξ) = n⁰_{bs} + ν_s ξ_{b…}.   n0: (B, S), ν: (S,), ξ: (B, …) → (B, …, S).  No index summed."""
    shape = (n0.shape[0],) + (1,) * (xi.dim() - 1) + (n0.shape[-1],)
    return n0.reshape(shape) + nu * xi[..., None]


def max_extent(n0: torch.Tensor, nu: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """ξ^max = min over reactants of n⁰_s / |ν_s|, and which reactant reaches zero first.   (…, S) → (…), (…).

    Products (ν_s > 0) never run out, so they get +∞ and can never be the minimum.
    """
    ratio = torch.where(nu < 0, n0 / nu.abs(), torch.inf)  # (…, S) how many "reaction units" each reactant allows
    xi_max, limiting = ratio.min(dim=-1)
    return xi_max, limiting


def theoretical_mass(xi_max: torch.Tensor, nu: torch.Tensor, molar: torch.Tensor, product: int = -1) -> torch.Tensor:
    """Most product the reaction can make: m_p = M_p ν_p ξ^max (g).   (…) → (…)."""
    return molar[product] * nu[product] * xi_max


def percent_yield(weighed: torch.Tensor, theoretical: torch.Tensor) -> torch.Tensor:
    """Actual over theoretical, in %.   (B,) → (B,)."""
    return 100.0 * weighed / theoretical


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/chemistry/limiting_reagent_yield/tensors.py
"""Build the concrete tensors: the reaction (stoichiometric numbers, molar masses) and a batch of experiments.

Units: masses in g, amounts in mol, molar masses in g/mol. Atomic masses are standard values (IUPAC, rounded).
"""

import torch

ATOMIC_MASS = {"H": 1.008, "N": 14.007, "Al": 26.982, "I": 126.904}

# Each reaction: two reactants then one product, with signed stoichiometric numbers ν_s (< 0 consumed, > 0 formed),
# the atoms in each species, and a few lab-notebook runs: (mass reactant 1, mass reactant 2, product weighed), in g.
REACTIONS = {
    "aluminium_iodine": {
        "species": ["Al", "I₂", "AlI₃"],
        "nu": [-2, -3, 2],  # 2 Al + 3 I₂ → 2 AlI₃
        "atoms": [{"Al": 1}, {"I": 2}, {"Al": 1, "I": 3}],
        "runs": [(0.50, 6.00, 4.10), (2.00, 2.00, 1.80), (1.00, 14.11, 12.90)],
    },
    "haber": {
        "species": ["N₂", "H₂", "NH₃"],
        "nu": [-1, -3, 2],  # N₂ + 3 H₂ → 2 NH₃
        "atoms": [{"N": 2}, {"H": 2}, {"N": 1, "H": 3}],
        "runs": [(28.0, 6.0, 5.1), (100.0, 10.0, 9.0), (28.0, 20.0, 6.2)],  # a single pass converts ~15 %
    },
}


def stoichiometry(reaction: str, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, list[str]]:
    """ν_s (S,), molar masses M_s (S,) and species names.   M_s = A_{es} m_e (summed over elements e)."""
    r = REACTIONS[reaction]
    elements = sorted(ATOMIC_MASS)
    A = torch.tensor([[a.get(e, 0) for a in r["atoms"]] for e in elements], dtype=torch.float64, device=device)
    m = torch.tensor([ATOMIC_MASS[e] for e in elements], dtype=torch.float64, device=device)
    molar = torch.einsum("es,e->s", A, m)  # (S,) g/mol
    nu = torch.tensor(r["nu"], dtype=torch.float64, device=device)
    return nu, molar, r["species"]


def experiments(reaction: str, mass_1: float, mass_2: float, measured: float, device: torch.device):
    """Starting masses of all species (B, S) in g — products start at 0 — and the product weighed (B,).

    The batch is the lab-notebook runs plus "your run" from the parameters as the LAST row.
    """
    runs = REACTIONS[reaction]["runs"] + [(mass_1, mass_2, measured)]
    m0 = torch.tensor([[a, b, 0.0] for a, b, _ in runs], dtype=torch.float64, device=device)
    weighed = torch.tensor([w for _, _, w in runs], dtype=torch.float64, device=device)
    return m0, weighed


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/chemistry/limiting_reagent_yield/solve.py
"""Stoichiometry for a whole batch at once: limiting reagent, yields, the extent sweep and the yield map."""

import torch

from . import equation


def analyse(m0: torch.Tensor, weighed: torch.Tensor, nu: torch.Tensor, molar: torch.Tensor) -> dict:
    """Everything for the batch of experiments: m0 (B, S) starting masses, weighed (B,) product masses."""
    n0 = equation.moles(m0, molar)  # (B, S)
    xi_max, limiting = equation.max_extent(n0, nu)  # (B,), (B,)
    n_end = equation.amounts(n0, nu, xi_max)  # (B, S): the limiting reagent is exactly 0
    theory = equation.theoretical_mass(xi_max, nu, molar)  # (B,)
    return {
        "n0": n0,
        "xi_max": xi_max,
        "limiting": limiting,
        "n_end": n_end,
        "leftover_mass": n_end * molar,  # (B, S) g of each species when the reaction stops
        "theory": theory,
        "percent": equation.percent_yield(weighed, theory),
    }


def extent_sweep(n0: torch.Tensor, nu: torch.Tensor, xi_max: torch.Tensor, steps: int = 5):
    """Amounts at ξ = 0, ¼, ½, ¾, 1 × ξ^max for every experiment.   → ξ (B, K), n (B, K, S)."""
    frac = torch.linspace(0.0, 1.0, steps, dtype=n0.dtype, device=n0.device)  # (K,)
    xi = xi_max[:, None] * frac  # (B, K)
    return xi, equation.amounts(n0, nu, xi)


def yield_map(mass_1: torch.Tensor, mass_2: torch.Tensor, nu: torch.Tensor, molar: torch.Tensor) -> torch.Tensor:
    """Theoretical product mass over a grid of the two reactant masses.   (G1,), (G2,) → (G2, G1) in g.

    The grid is just a big batch: n⁰ has shape (G2, G1, S) and the same min-over-reactants does the rest.
    """
    m2, m1 = torch.meshgrid(mass_2, mass_1, indexing="ij")  # (G2, G1)
    m0 = torch.stack([m1, m2, torch.zeros_like(m1)], dim=-1)  # (G2, G1, S)
    xi_max, _ = equation.max_extent(equation.moles(m0, molar), nu)
    return equation.theoretical_mass(xi_max, nu, molar)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/chemistry/limiting_reagent_yield/plot.py
"""Static figure: amounts against reaction extent, the yield map over both masses, and the batch of runs."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import save_png


def figure(
    xi: torch.Tensor,
    n_sweep: torch.Tensor,
    species: list[str],
    res: dict,
    weighed: torch.Tensor,
    grid_1: torch.Tensor,
    grid_2: torch.Tensor,
    product_map: torch.Tensor,
    ratio: torch.Tensor,
    m0: torch.Tensor,
    out_dir: Path,
) -> Path:
    fig, (ax_x, ax_m, ax_b) = plt.subplots(1, 3, figsize=(13, 4.3), gridspec_kw={"width_ratios": [1.1, 1.05, 1]})

    # --- (a) bars of every amount at ξ = 0 … ξ_max (your run) ---
    x, n = xi.cpu().numpy(), n_sweep.cpu().numpy() * 1e3  # mmol
    K, S = n.shape
    width = 0.8 / S
    for s in range(S):
        ax_x.bar(np.arange(K) + (s - (S - 1) / 2) * width, n[:, s], width, color=f"C{s}", label=species[s])
    lim = res["limiting"][-1].item()
    ax_x.set_xticks(range(K), [f"{f:.2g}·ξmax\n{v * 1e3:.3g}" for f, v in zip(np.linspace(0, 1, K), x)], fontsize=8)
    ax_x.set_xlabel("reaction extent ξ (mmol)")
    ax_x.set_ylabel("amount n (mmol)")
    ax_x.set_title(f"your run: {species[lim]} runs out first (limiting)", fontsize=10)
    ax_x.legend(fontsize=8)

    # --- (b) theoretical product over both masses, with the stoichiometric line ---
    g1, g2, pm = grid_1.cpu().numpy(), grid_2.cpu().numpy(), product_map.cpu().numpy()
    levels = np.linspace(0, pm.max(), 11)
    filled = ax_m.contourf(g1, g2, pm, levels=levels, cmap="Blues")
    fig.colorbar(filled, ax=ax_m, label=f"theoretical {species[2]} (g)")
    ax_m.plot(g1, ratio.item() * g1, "--", color="#dc2626", lw=1.8, label="stoichiometric ratio")
    ax_m.plot(m0[-1, 0].item(), m0[-1, 1].item(), "o", ms=8, mfc="#f59e0b", mec="#111827", label="your run")
    box = {"facecolor": "white", "edgecolor": "none", "alpha": 0.8}
    ax_m.text(0.72, 0.28, f"{species[1]} limiting", ha="center", transform=ax_m.transAxes, fontsize=8, bbox=box)
    ax_m.text(0.28, 0.72, f"{species[0]} limiting", ha="center", transform=ax_m.transAxes, fontsize=8, bbox=box)
    ax_m.set_xlim(g1[0], g1[-1])
    ax_m.set_ylim(g2[0], g2[-1])
    ax_m.set_xlabel(f"mass of {species[0]} (g)")
    ax_m.set_ylabel(f"mass of {species[1]} (g)")
    ax_m.set_title("yield map: contours bend at the line", fontsize=10)
    ax_m.legend(fontsize=7.5, loc="upper left")
    ax_m.grid(False)

    # --- (c) the batch: theoretical vs weighed product, percent yield on top ---
    B = len(weighed)
    theory, pct = res["theory"].cpu().numpy(), res["percent"].cpu().numpy()
    names = [f"run {b + 1}" for b in range(B - 1)] + ["your run"]
    ax_b.bar(np.arange(B) - 0.2, theory, 0.4, color="#93c5fd", edgecolor="#1e40af", label="theoretical")
    ax_b.bar(np.arange(B) + 0.2, weighed.cpu().numpy(), 0.4, color="#2563eb", label="weighed")
    for b in range(B):
        limiting = species[res["limiting"][b].item()]
        top = max(theory[b], weighed[b].item())
        ax_b.text(b, top * 1.03, f"{pct[b]:.0f} %\n({limiting} lim.)", ha="center", fontsize=8)
    ax_b.set_xticks(range(B), names)
    ax_b.set_ylim(0, 1.25 * max(theory.max(), weighed.max().item()))
    ax_b.set_ylabel(f"{species[2]} (g)")
    ax_b.set_title("percent yield = weighed / theoretical", fontsize=10)
    ax_b.legend(fontsize=8, loc="upper left")

    fig.tight_layout()
    return save_png(fig, out_dir / "limiting_reagent_yield.png")


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'reaction': 'aluminium_iodine',
    'mass_1': 1.2,
    'mass_2': 2.4,
    'measured': 2.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.chemistry.limiting_reagent_yield")]:
    del sys.modules[name]
main = importlib.import_module("problems.chemistry.limiting_reagent_yield.main")
problem_dir = Path("problems/chemistry/limiting_reagent_yield")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
